In [ ]:
import os
import random
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as ticker
from scipy.stats import t
import matplotlib.patches as mpatches

# === 全局样式修正：使用 DejaVu Sans 替代 Helvetica ===
mpl.rc('xtick', direction='out', labelsize=8)
mpl.rc('ytick', direction='out', labelsize=8)
mpl.rcParams['xtick.major.width'] = 0.8
mpl.rcParams['ytick.major.width'] = 0.8
mpl.rc('font', family='sans-serif')
mpl.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'Liberation Sans']
mpl.rc('axes', titlesize=12, labelsize=10, linewidth=0.8)
mpl.rc('legend', fontsize=8, frameon=False)
mpl.rc('figure', figsize=(6,4), dpi=300)
mpl.rc('savefig', bbox='tight', pad_inches=0.1)
# === 路径设置 ===
MERRA2_FILE = "/mnt/backup_ETH/Marina/MERRA2/daily_fields/MERRA2_3d_dm_r144x96_1980-05.2020_SLP_O3.nc"
OUTPUT_DIR  = '/home/weiji/restart_exam/plots/MERRA2_O3/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

paths = {
    'anom_kg'    : os.path.join(OUTPUT_DIR, "MERRA2_O3_Anomaly_2020.png"),
    'clim_kg'    : os.path.join(OUTPUT_DIR, "MERRA2_O3_Climatology.png"),
    'clim_ppm'   : os.path.join(OUTPUT_DIR, "MERRA2_O3_Climatology_ppm.png"),
    'anom_ppm'   : os.path.join(OUTPUT_DIR, "MERRA2_O3_Anomaly_2020_ppm.png"),
    't_kg'       : os.path.join(OUTPUT_DIR, "MERRA2_O3_Anomaly2020_ttest_kgkg.png"),
    't_ppm'      : os.path.join(OUTPUT_DIR, "MERRA2_O3_Anomaly2020_ttest_ppm.png"),
    'b_kg'       : os.path.join(OUTPUT_DIR, "MERRA2_O3_Anomaly2020_boot_kgkg.png"),
    'b_ppm'      : os.path.join(OUTPUT_DIR, "MERRA2_O3_Anomaly2020_boot_ppm.png"),
    # —— 新增“不显著”：
    'ns_b_ppm'    : os.path.join(OUTPUT_DIR, "MERRA2_O3_Anomaly2020_nonsig_b_ppm.png"),
    'ns_t_ppm'   : os.path.join(OUTPUT_DIR, "MERRA2_O3_Anomaly2020_nonsig_t_ppm.png"),
}

# === 常量 & 参数 ===
M_air, M_O3 = 28.97, 48.0    # g/mol
conv        = M_air / M_O3
MONTHS      = range(1,6)
BASE_YRS    = range(1980,2020)

# === 数据处理（与之前完全相同） ===
def process_o3(da):
    da = da.mean(dim='lon')
    da = da.sel(lat=slice(60,90))
    w  = np.cos(np.deg2rad(da.lat))
    return da.weighted(w).mean(dim='lat')

def bootstrap_ci(data, nboot=500, alpha=0.05):
    means = np.empty(nboot)
    n = data.size
    for i in range(nboot):
        samp = np.random.choice(data, size=n, replace=True)
        means[i] = samp.mean()
    return np.percentile(means, [100*alpha/2, 100*(1-alpha/2)])

# === 绘图函数（新增 show_nonsig） ===
def plot_merra2_o3_anomaly(anom_da, clim_da, save_path,
                           unit='kg/kg', sig_mask=None,
                           show_nonsig=False):
    fig, ax = plt.subplots()
    tnum = mdates.date2num(anom_da.time.values)
    p_pa = anom_da.lev.values * 100

    # 异常背景
    levels = np.linspace(-1.5, 1.5, 31) if unit=='ppm' else 20
    cf = ax.contourf(
        tnum, p_pa, anom_da.values,
        levels=levels, cmap='RdBu_r',
        extend='both', antialiased=True, alpha=0.85
    )

    # 气候态等值线
    clim_vals = clim_da.values
    if unit=='kg/kg':
        clim_lvls = [x/conv for x in [4.8e-6,5.2e-6,5.6e-6,6.0e-6,6.4e-6]]
        fmt = '%.1e'
    else:
        clim_lvls = [4.8,5.2,5.6,6.0,6.4]
        fmt = '%.1f'
    CS = ax.contour(
        tnum, p_pa, clim_vals,
        levels=clim_lvls, colors='k', linewidths=1.0
    )
    ax.clabel(CS, inline=True, fontsize=8, fmt=fmt)

    # 斜线花纹：显著 or 不显著
    if sig_mask is not None:
        if show_nonsig:
            mask       = ~sig_mask
            legend_txt = 'Not significant (p ≥ 0.05)'
        else:
            mask       =  sig_mask
            legend_txt = 'Significant (p < 0.05)'

        sig_da = xr.DataArray(
            mask.astype(int),
            coords=anom_da.coords,
            dims=anom_da.dims
        )
        ax.contourf(
            tnum, p_pa, sig_da,
            levels=[0.5,1.5],
            colors='none',
            hatches=['///'],
            alpha=0
        )
        # legend patch
        patch = mpatches.Patch(facecolor='white',
                               hatch='///',
                               edgecolor='black',
                               label=legend_txt)
        ax.legend(handles=[patch], loc='upper right')

    # 坐标格式
    ax.set_yscale('log')
    ax.invert_yaxis()
    ax.set_xlim(tnum[0], tnum[-1])
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    ax.grid(True, which='major', linestyle='--', linewidth=0.4, alpha=0.5)
    ax.tick_params(top=False, right=False)

    ax.set_ylabel('Pressure (Pa)')
    ax.set_title('MERRA2 O$_3$ Anomaly (2020) with Climatology')

    cax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
    fig.colorbar(cf, cax=cax, label=f'O$_3$ Anomaly ({unit})')

    plt.savefig(save_path, dpi=150)
    plt.show()

# === 主流程 ===
ds    = xr.open_dataset(MERRA2_FILE)
o3all = ds['O3']

# 1. 气候态
da_clim_full = o3all.sel(
    time=(o3all.time.dt.month.isin(MONTHS)) &
         (o3all.time.dt.year.isin(BASE_YRS))
)
da_clim = da_clim_full.groupby('time.dayofyear').mean(dim='time')
da_clim = process_o3(da_clim)

# 2. 2020 数据
da_2020 = o3all.sel(
    time=(o3all.time.dt.year==2020) &
         (o3all.time.dt.month.isin(MONTHS))
)
da_2020 = process_o3(da_2020)

# 3. 对齐
times20 = da_2020.time.values
doy20   = da_2020.time.dt.dayofyear.values
da_clim_aligned = da_clim.sel(dayofyear=doy20)
da_clim_aligned = xr.DataArray(
    da_clim_aligned.values.T,
    coords={'lev': da_clim.lev.values, 'time': times20},
    dims=['lev','time']
)

# 4. 异常
anom_da = xr.DataArray(
    da_2020.values.T - da_clim_aligned.values,
    coords={'lev': da_2020.lev.values, 'time': times20},
    dims=['lev','time']
)

# 5. 单位转换
anom_plot_da_kg  = anom_da
anom_plot_da_ppm = anom_da * conv * 1e6
clim_plot_da_kg  = da_clim_aligned
clim_plot_da_ppm = da_clim_aligned * conv * 1e6

# 6. 统计显著性
lev_n, time_n = anom_da.sizes['lev'], anom_da.sizes['time']
t_mask = np.zeros((lev_n, time_n), bool)
b_mask = np.zeros((lev_n, time_n), bool)

da_base = o3all.sel(
    time=(o3all.time.dt.month.isin(MONTHS)) &
         (o3all.time.dt.year.isin(BASE_YRS))
)
da_base = process_o3(da_base)
times_b = da_base.time.values
doy_b   = da_base.time.dt.dayofyear.values
da_clim_b = da_clim.sel(dayofyear=doy_b)
da_clim_b = xr.DataArray(
    da_clim_b.values.T,
    coords={'lev': da_clim.lev.values, 'time': times_b},
    dims=['lev','time']
)
base_anom = xr.DataArray(
    da_base.values.T - da_clim_b.values,
    coords=da_clim_b.coords,
    dims=da_clim_b.dims
)

for ti, tval in enumerate(times20):
    doy = pd.Timestamp(tval).dayofyear
    samp = base_anom.sel(time=base_anom.time.dt.dayofyear==doy).values
    for li in range(lev_n):
        col = samp[li, :]
        col = col[~np.isnan(col)]
        if col.size < 2:
            continue
        obs = anom_da.values[li, ti]
        se    = np.std(col, ddof=1)/np.sqrt(col.size)
        tstat = obs/se
        pval  = 2*(1 - t.cdf(abs(tstat), df=col.size-1))
        t_mask[li, ti] = (pval < 0.05)
        lo, hi = bootstrap_ci(col)
        b_mask[li, ti] = not (lo <= obs <= hi)

# === 原有图 ===
#plot_merra2_o3_anomaly(anom_plot_da_kg,  clim_plot_da_kg,  paths['anom_kg'],  unit='kg/kg')
plot_merra2_o3_anomaly(anom_plot_da_ppm, clim_plot_da_ppm, paths['anom_ppm'], unit='ppm')
#plot_merra2_o3_anomaly(anom_plot_da_kg,  clim_plot_da_kg,  paths['t_kg'],      unit='kg/kg', sig_mask=t_mask)
plot_merra2_o3_anomaly(anom_plot_da_ppm, clim_plot_da_ppm, paths['t_ppm'],     unit='ppm',    sig_mask=t_mask)
#plot_merra2_o3_anomaly(anom_plot_da_kg,  clim_plot_da_kg,  paths['b_kg'],      unit='kg/kg', sig_mask=b_mask)
plot_merra2_o3_anomaly(anom_plot_da_ppm, clim_plot_da_ppm, paths['b_ppm'],     unit='ppm',    sig_mask=b_mask)

# === 新增“非显著区”对比图 ===
plot_merra2_o3_anomaly(anom_plot_da_ppm,  clim_plot_da_ppm,  paths['ns_b_ppm'],  unit='ppm', sig_mask=b_mask, show_nonsig=True)
plot_merra2_o3_anomaly(anom_plot_da_ppm, clim_plot_da_ppm, paths['ns_t_ppm'], unit='ppm',    sig_mask=t_mask, show_nonsig=True)

print("所有图（含显著／不显著）均已生成并展示。")
